In [1]:
import os
import pandas as pd
import re

In [25]:
header = ["execid", "benchmark", "instance", "executable", "cost", "status", "exit_code", "real", "time", "user", "system", "memory"]

result_dir = "results"
df = None
for file in os.listdir(result_dir):
    if not file.endswith(".tsv"):
        continue
    file = f"{result_dir}/{file}"
    df_curr = pd.read_csv(file, sep="\t", names=header, index_col=False)
    df = pd.concat([df, df_curr]) if not df is None else df_curr
 
df.insert(len(header), "win", False)
df.insert(len(header)+1, "score", 0)
df["cost"] = df["cost"].str.replace(",","").astype(float)
df["exit_code"] = df["exit_code"].astype(int)
df["O"] = df["exit_code"] == 30
df

,execid,benchmark,instance,executable,cost,status,exit_code,real,time,user,system,memory,win,score,O
0,nomin,GraphColouring,0001-graph_colouring-125-0_1200.asp,clingo-plainmaximize-amo,2076.0,outof time,11,"1,200.56","1,200.29","1,200.02",0.27,142.80,False,0,False
1,nomin,GraphColouring,0001-graph_colouring-125-0_2400.asp,clingo-plainmaximize-amo,2076.0,outof time,11,"1,200.45","1,200.37","1,200.16",0.21,119.70,False,0,False
2,nomin,GraphColouring,0001-graph_colouring-125-0_3600.asp,clingo-plainmaximize-amo,2076.0,outof time,11,"1,200.49","1,200.14","1,199.80",0.34,119.50,False,0,False
3,nomin,GraphColouring,0001-graph_colouring-125-0_4800.asp,clingo-plainmaximize-amo,2076.0,outof time,11,"1,200.25","1,200.14","1,199.29",0.85,118.70,False,0,False
4,nomin,GraphColouring,0001-graph_colouring-125-0_6000.asp,clingo-plainmaximize-amo,2076.0,outof time,11,"1,201.14","1,200.95","1,200.62",0.33,119.50,False,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,nomin,TSP,095-490-tsp.asp,clingo-amomaximize-amo-nomin,26599.0,outof time,29,"1,205.49","1,201.04","1,179.25",21.79,"3,060.90",False,0,False
1296,nomin,TSP,096-495-tsp.asp,clingo-amomaximize-amo-nomin,26421.0,outof time,29,"1,206.32","1,200.95","1,173.00",27.95,"3,242.80",False,0,False
1297,nomin,TSP,097-500-tsp.asp,clingo-amomaximize-amo-nomin,25988.0,outof time,10,"1,207.60","1,201.07","1,179.29",21.78,"3,091.30",False,0,False
1298,nomin,TSP,098-505-tsp.asp,clingo-amomaximize-amo-nomin,26987.0,outof time,29,"1,207.51","1,201.08","1,175.63",25.45,"3,163.60",False,0,False


In [30]:
from typing import Dict, List


def _filterDF(df: pd.DataFrame, filterList: List[Dict[str, str]] = {}, positive=True):
    for filter in filterList:
        sliceDf = None
        regexStr = ""
        for column in filter:
            regex = filter[column]
            notStr = "~" if not positive else ""
            regexStr = f" and {notStr}{regex}" if regexStr else regex
            sliceDf = (df[column].str.contains(regex)) if sliceDf is None else sliceDf & (df[column].str.contains(regex))
        print(f"{'Keeping' if positive else 'Removing'} {regexStr} from {column}")
        if positive:
            df = df[sliceDf]
        else:
            df = df[~sliceDf]

    return df


def filterDF(df: pd.DataFrame, positiveFilter: List[Dict[str, str]] = {}, negativeFilter: List[Dict[str, str]] = {}):
    df = _filterDF(df, positiveFilter, True)
    df = _filterDF(df, negativeFilter, False)
    return df


positiveFilter = [
]

negativeFilter = [
{
    "benchmark": r"Knapsack$",
}
]

df = filterDF(df, positiveFilter, negativeFilter)

Removing Knapsack$ from benchmark


In [32]:
df["benchmark"] = df["benchmark"].str.replace("KnapsackAmomaximize", "KC")
df["benchmark"] = df["benchmark"].str.replace("GraphColouring", "GC")

/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/ipykernel_13512/1506806122.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["benchmark"] = df["benchmark"].str.replace("KnapsackAmomaximize", "KC")
/var/folders/60/msx2m7995xv41v59mx28gt

In [26]:
# Win
groubByInstance = df.groupby("instance")

for instance, groupedDf in groubByInstance:
    max_cost = groupedDf["cost"].max()
    instance_mask = (df["instance"] == instance)
    df.loc[instance_mask & (df["cost"] == max_cost), "win"] = True
    df.loc[instance_mask, "score"] = df[instance_mask]["cost"]/max_cost


/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/ipykernel_13512/3181317159.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.         0.99962089]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[instance_mask, "score"] = df[instance_mask]["cost"]/max_cost


In [33]:
pivot = df.pivot_table(index=["executable"],columns=["benchmark"], values=["win","O", "score"], aggfunc={"win":"sum", "O": "sum", "score": "mean"}, margins=True, margins_name='Total', fill_value=-1)
pivot = pivot[:-1]
pivot

/opt/homebrew/anaconda3/envs/amosum/lib/python3.14/site-packages/pandas/core/frame.py:4343: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self[k1] = value[k2]
/opt/homebrew/anaconda3/envs/amosum/lib/python3.14/site-packages/pandas/core/resha

O                          score            \
benchmark                      GC   KC  TSP  WOD Total        GC        KC   
executable                                                                   
clingo-amomaximize-amo-nomin  0.0  0.0  5.0  6.0    11  0.999855  0.911893   
clingo-plainmaximize-amo      0.0  0.0  2.0  0.0     2  0.884748  0.995741   

                                                              win              \
benchmark                          TSP       WOD     Total     GC    KC   TSP   
executable                                                                      
clingo-amomaximize-amo-nomin  0.994959  0.702867  0.934881  289.0  21.0  82.0   
clingo-plainmaximize-amo      0.941614  0.968785  0.926731   11.0  79.0  20.0   

                                          
benchmark                      WOD Total  
executable                                
clingo-amomaximize-amo-nomin  53.0   445  
clingo-plainmaximize-amo      92.0   202

In [22]:
pivot = df.pivot_table(index=["executable"],columns=["benchmark"], values=["win"], aggfunc={"win":"sum"}, margins=True, margins_name='Total', fill_value=-1)
pivot.columns = [col[1] for col in pivot.columns]
pivot = pivot.reset_index()

def bold_max(col):
    if col.name == "executable":
        return col
    # find max excluding Total row
    max_val = col[:-1].max()
    return col.apply(lambda x: f"\\textbf{{{x}}}" if x == max_val else str(x))

styled = pivot.apply(bold_max)
# Convert to latex
table_latex = styled.to_latex(index=False, escape=False)



In [23]:
import subprocess
from pathlib import Path

def create_latex_visualization(latex_input: str, output_dir: str, output_name, tex_name: str):
    latex = rf"""
    \documentclass{{standalone}}
    \usepackage{{booktabs}}
    \begin{{document}}

    {latex_input}

    \end{{document}}
    """

    output_path = Path(f"{output_dir}/{output_name}")
    output_path.mkdir(parents=True, exist_ok=True)

    tex_name = f"{tex_name}.tex"
    tex_file = output_path / f"{tex_name}"
    tex_file.write_text(latex)

    subprocess.run(
        ["pdflatex", tex_name],
        cwd=output_path
    )

In [24]:
create_latex_visualization(table_latex, "tables", "nominres", "table")

This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./table.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2025/02/22 v1.5a Class to compile TeX sub-files stan
dalone
(/usr/local/texlive/2025/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/local/texlive/2025/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/local/texlive/2025/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/local/texlive/2025/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/local/texlive/2025/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/local/texlive/2025/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/local/texlive/2025/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/local/texlive/2025/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/article.cls
Docum